# 🚀 Advanced Jigsaw Models: State-of-the-Art Techniques (FIXED)

## 🎯 Advanced Techniques:
1. **Transformer Fine-tuning**: BERT, RoBERTa
2. **Meta-Learning**: MAML for unseen rule adaptation
3. **Enhanced Ensembling**: Stacking with meta-learners


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
import re
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔥 Using device: {device}')

# Try MAML import
try:
    import learn2learn as l2l
    MAML_AVAILABLE = True
    print('✅ MAML available')
except ImportError:
    MAML_AVAILABLE = False
    print('⚠️ MAML not available')

🔥 Using device: cuda
✅ MAML available


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Load datasets
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')
print(f'📊 Training: {train_df.shape}, Test: {test_df.shape}')

# Text preprocessing
def clean_text(text):
    if pd.isna(text): return ''
    text = str(text).strip()
    text = re.sub(r'http[s]?://\S+', '[URL]', text)
    text = re.sub(r'u/\w+', '[USER]', text)
    text = re.sub(r'r/\w+', '[SUBREDDIT]', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df['processed_text'] = train_df['body'].apply(clean_text)
test_df['processed_text'] = test_df['body'].apply(clean_text)
train_df['rule_aware_text'] = train_df['rule'] + ' [SEP] ' + train_df['processed_text']
test_df['rule_aware_text'] = test_df['rule'] + ' [SEP] ' + test_df['processed_text']
print('✅ Text preprocessing completed!')

📊 Training: (2029, 9), Test: (10, 8)
✅ Text preprocessing completed!


In [4]:
class TransformerClassifier:
    def __init__(self, model_name, max_length=512):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = None
        special_tokens = ['[URL]', '[USER]', '[SUBREDDIT]', '[SEP]']
        self.tokenizer.add_tokens(special_tokens)
    
    def prepare_dataset(self, texts, labels=None):
        encodings = self.tokenizer(texts, truncation=True, padding=True, 
                                 max_length=self.max_length, return_tensors='pt')
        dataset_dict = {
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask']
        }
        if labels is not None:
            dataset_dict['labels'] = torch.tensor(labels, dtype=torch.long)
        return Dataset.from_dict({k: v.tolist() if hasattr(v, 'tolist') else v 
                                 for k, v in dataset_dict.items()})
    
    def train(self, train_texts, train_labels, val_texts, val_labels):
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=2)
        self.model.resize_token_embeddings(len(self.tokenizer))
        self.model.to(device)
        
        train_dataset = self.prepare_dataset(train_texts, train_labels)
        val_dataset = self.prepare_dataset(val_texts, val_labels)
        
        training_args = TrainingArguments(
            output_dir=f'./results_{self.model_name.replace("/", "_")}',
            num_train_epochs=2, per_device_train_batch_size=8,
            per_device_eval_batch_size=16, warmup_steps=100,
            weight_decay=0.01, eval_strategy="steps", eval_steps=200,
            save_strategy="steps", save_steps=200,
            load_best_model_at_end=True, fp16=torch.cuda.is_available(),
            dataloader_num_workers=0, remove_unused_columns=False
        )
        trainer = Trainer(model=self.model, args=training_args,
                         train_dataset=train_dataset, eval_dataset=val_dataset)
        trainer.train()
        return trainer
    
    def predict(self, texts):
        dataset = self.prepare_dataset(texts)
        self.model.eval()
        predictions = []
        with torch.no_grad():
            def collate_fn(batch):
                collated = {}
                for key in batch[0].keys():
                    if key != 'labels':
                        values = [item[key] for item in batch]
                        collated[key] = torch.tensor(values).to(device)
                return collated
            dataloader = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
            for batch in dataloader:
                outputs = self.model(**batch)
                probs = torch.softmax(outputs.logits, dim=-1)
                predictions.extend(probs[:, 1].cpu().numpy())
        return np.array(predictions)

print('🤖 Transformer classifier ready!')

🤖 Transformer classifier ready!


In [5]:
# FIXED: Properly execute transformer training
transformer_models = {'bert': 'bert-base-uncased', 'roberta': 'roberta-base'}

# Split data FIRST
X = train_df['rule_aware_text'].tolist()
y = train_df['rule_violation'].tolist()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"📊 Training: {len(X_train)}, Validation: {len(X_val)}")

# Initialize containers BEFORE training
transformer_predictions = {}
trained_models = {}

# Train each transformer
for name, model_name in transformer_models.items():
    try:
        print(f'\n🤖 Training {name} ({model_name})...')
        classifier = TransformerClassifier(model_name)
        trainer = classifier.train(X_train, y_train, X_val, y_val)
        
        # Get validation predictions
        val_preds = classifier.predict(X_val)
        val_auc = roc_auc_score(y_val, val_preds)
        
        # Store results
        transformer_predictions[name] = {'val_preds': val_preds, 'val_auc': val_auc}
        trained_models[name] = classifier
        
        print(f'✅ {name} completed! Validation AUC: {val_auc:.4f}')
        
    except Exception as e:
        print(f'❌ {name} failed: {str(e)}')
        continue

print(f'\n✅ Successfully trained {len(trained_models)} transformer models')
for name, results in transformer_predictions.items():
    print(f'   • {name}: {results["val_auc"]:.4f} AUC')

📊 Training: 1623, Validation: 406

🤖 Training bert (bert-base-uncased)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Step,Training Loss,Validation Loss
200,No log,0.480368
400,No log,0.470550


✅ bert completed! Validation AUC: 0.8922

🤖 Training roberta (roberta-base)...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss
200,No log,0.482725
400,No log,0.466546


✅ roberta completed! Validation AUC: 0.8804

✅ Successfully trained 2 transformer models
   • bert: 0.8922 AUC
   • roberta: 0.8804 AUC


In [6]:
# MAML Meta-Learning (if available)
if MAML_AVAILABLE:
    print('🧠 Training MAML...')
    
    class MAMLClassifier(nn.Module):
        def __init__(self, input_dim, hidden_dim=256):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(hidden_dim // 2, 2)
            )
        def forward(self, x): return self.network(x)

    try:
        sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
        train_embeddings = sentence_model.encode(X_train, show_progress_bar=True)
        val_embeddings = sentence_model.encode(X_val, show_progress_bar=True)

        input_dim = train_embeddings.shape[1]
        maml_model = MAMLClassifier(input_dim).to(device)
        maml = l2l.algorithms.MAML(maml_model, lr=0.01)
        meta_optimizer = torch.optim.Adam(maml.parameters(), lr=0.001)
        loss_fn = nn.CrossEntropyLoss()

        train_embeddings_tensor = torch.FloatTensor(train_embeddings).to(device)
        train_labels_tensor = torch.LongTensor(y_train).to(device)

        for epoch in range(10):
            meta_optimizer.zero_grad()
            meta_loss = 0.0
            
            for _ in range(2):
                task_indices = np.random.choice(len(train_embeddings), size=32, replace=False)
                task_x = train_embeddings_tensor[task_indices]
                task_y = train_labels_tensor[task_indices]
                
                support_x, query_x = task_x[:16], task_x[16:]
                support_y, query_y = task_y[:16], task_y[16:]
                
                learner = maml.clone()
                support_preds = learner(support_x)
                support_loss = loss_fn(support_preds, support_y)
                learner.adapt(support_loss)
                
                query_preds = learner(query_x)
                query_loss = loss_fn(query_preds, query_y)
                meta_loss += query_loss
            
            meta_loss /= 2
            meta_loss.backward()
            meta_optimizer.step()
            
            if epoch % 5 == 0:
                print(f'MAML Epoch {epoch}, Loss: {meta_loss.item():.4f}')

        print('✅ MAML training completed!')
        MAML_TRAINED = True
        
    except Exception as e:
        print(f'❌ MAML failed: {str(e)}')
        MAML_TRAINED = False
        maml_model = None
        sentence_model = None
        val_embeddings = None
else:
    print('⚠️ MAML not available')
    MAML_TRAINED = False
    maml_model = None
    sentence_model = None
    val_embeddings = None

🧠 Training MAML...


Batches: 100%|██████████| 13/13 [00:00<00:00, 51.67it/s]


MAML Epoch 0, Loss: 0.6760
MAML Epoch 5, Loss: 0.7092
✅ MAML training completed!


In [7]:
# FIXED: Enhanced ensemble with proper variable handling
print('🎯 Creating enhanced ensemble...')

base_predictions = []
model_names = []

# Add transformer predictions (now properly defined)
for name, results in transformer_predictions.items():
    base_predictions.append(results['val_preds'])
    model_names.append(f'transformer_{name}')

# Add MAML predictions if available
if MAML_TRAINED and maml_model is not None and val_embeddings is not None:
    try:
        val_embeddings_tensor = torch.FloatTensor(val_embeddings).to(device)
        with torch.no_grad():
            maml_outputs = maml_model(val_embeddings_tensor)
            maml_probs = torch.softmax(maml_outputs, dim=-1)
            maml_val_preds = maml_probs[:, 1].cpu().numpy()
        base_predictions.append(maml_val_preds)
        model_names.append('maml')
        print('✅ MAML predictions added')
    except Exception as e:
        print(f'⚠️ MAML prediction failed: {str(e)}')

# Create ensemble
if len(base_predictions) > 0:
    stacking_features = np.column_stack(base_predictions)
    print(f'📊 Stacking features shape: {stacking_features.shape}')

    meta_learner = LogisticRegression(random_state=42)
    meta_learner.fit(stacking_features, y_val)

    ensemble_preds = meta_learner.predict_proba(stacking_features)[:, 1]
    ensemble_auc = roc_auc_score(y_val, ensemble_preds)

    print(f'🎯 Enhanced Ensemble AUC: {ensemble_auc:.4f}')

    importance_df = pd.DataFrame({
        'model': model_names,
        'importance': np.abs(meta_learner.coef_[0])
    }).sort_values('importance', ascending=False)

    print('\n📊 Model Importance:')
    for _, row in importance_df.iterrows():
        print(f'   • {row["model"]}: {row["importance"]:.4f}')
else:
    print('❌ No predictions for ensemble')
    ensemble_auc = 0.0
    meta_learner = None

🎯 Creating enhanced ensemble...
✅ MAML predictions added
📊 Stacking features shape: (406, 3)
🎯 Enhanced Ensemble AUC: 0.8926

📊 Model Importance:
   • transformer_roberta: 2.0782
   • transformer_bert: 2.0428
   • maml: 0.0629


In [8]:
# Generate test predictions and create submission
if len(trained_models) > 0 and meta_learner is not None:
    print('🎯 Generating test predictions...')

    test_texts = test_df['rule_aware_text'].tolist()
    test_predictions_all = []

    # Get transformer test predictions
    for name, model in trained_models.items():
        try:
            test_preds = model.predict(test_texts)
            test_predictions_all.append(test_preds)
            print(f'✅ {name} test predictions generated')
        except Exception as e:
            print(f'❌ {name} test failed: {str(e)}')

    # MAML test predictions
    if MAML_TRAINED and maml_model is not None and sentence_model is not None:
        try:
            test_embeddings = sentence_model.encode(test_texts)
            test_embeddings_tensor = torch.FloatTensor(test_embeddings).to(device)
            with torch.no_grad():
                maml_test_outputs = maml_model(test_embeddings_tensor)
                maml_test_probs = torch.softmax(maml_test_outputs, dim=-1)
                maml_test_preds = maml_test_probs[:, 1].cpu().numpy()
            test_predictions_all.append(maml_test_preds)
            print('✅ MAML test predictions generated')
        except Exception as e:
            print(f'⚠️ MAML test failed: {str(e)}')

    # Create ensemble test predictions
    if len(test_predictions_all) > 0:
        test_stacking_features = np.column_stack(test_predictions_all)
        final_test_preds = meta_learner.predict_proba(test_stacking_features)[:, 1]

        # Create submission
        submission_df = pd.DataFrame({
            'row_id': range(len(test_df)),
            'rule_violation': final_test_preds
        })
        submission_df.to_csv('submission_advanced_ensemble_fixed.csv', index=False)
        print('✅ Advanced ensemble submission saved!')

        # Summary
        print('\n' + '='*60)
        print('🚀 ADVANCED MODELS - FINAL SUMMARY (FIXED)')
        print('='*60)
        print(f'📊 Models trained: {len(model_names)}')
        print(f'🎯 Enhanced Ensemble AUC: {ensemble_auc:.4f}')
        print('📁 Output: submission_advanced_ensemble_fixed.csv')
        print(' MODELING COMPLETED!')
        print('='*60)
    else:
        print('❌ No test predictions generated')
else:
    print('❌ No trained models available for test predictions')

🎯 Generating test predictions...
✅ bert test predictions generated
✅ roberta test predictions generated
✅ MAML test predictions generated
✅ Advanced ensemble submission saved!

🚀 ADVANCED MODELS - FINAL SUMMARY (FIXED)
📊 Models trained: 3
🎯 Enhanced Ensemble AUC: 0.8926
📁 Output: submission_advanced_ensemble_fixed.csv
 MODELING COMPLETED!
